# Data Preprocessing
In this Notebook we perform data cleaning and Preparation for Multi-Omics Analysis

#### Import Packages

In [53]:
import pandas as pd
from tqdm import tqdm
import warnings
warnings.filterwarnings("ignore")

### Load Data

In [54]:
df1 = pd.read_csv("../downloads/ROSMAP/ROSMAP_arrayMethylation_metaData.tsv", sep = "\t")
df1=df1.dropna()
iD_to_loc = dict(zip(df1.TargetID, df1.CpG_Island))

In [56]:
import pandas as pd

input_path = "../data/ROSMAP/methylation_data.csv"
output_path = "../data/ROSMAP/methylation_data_locations.csv"

full_header = pd.read_csv(input_path, nrows=0).columns.tolist()

first_chunk = True
valid_cols = [col for col in full_header if col in iD_to_loc]

chunks = pd.read_csv(input_path, chunksize=5, skip_blank_lines=False, keep_default_na=False)
for chunk in tqdm(chunks):
    
    filtered_chunk = chunk[["Sample ID", "Gender", "Race", "PMI", "Braak", "Diagnosis"] + valid_cols]
    
    filtered_chunk = filtered_chunk.rename(columns=iD_to_loc)
    
    # Save incrementally
    if first_chunk:
        filtered_chunk.to_csv(output_path, index=False)
        first_chunk = False
    else:
        filtered_chunk.to_csv(output_path, mode="a", header=False, index=False)

31it [04:45,  9.20s/it]


### Load Graph and Prepare Enhancer and Promoter Data

In [2]:
GRAPH_PATH = "../artifacts/universalGraph_new.json"
with open(GRAPH_PATH, "r") as f:
    graph_json = json.load(f)

In [3]:
graph_json.keys()

dict_keys(['vertices', 'edges', 'directed'])

In [59]:
# # iterate over vertices
# Enhancer_Promoters = {"Enhancer": {}, "Promoter":{}, "Promoter/Enhancer":{}}

# for vertex, vertex_data in graph_json["vertices"].items():
#     omic_type = vertex_data["omic_type"]
#     if omic_type in Enhancer_Promoters:
#         position = vertex_data['vert_info']["hg19"]
#         label = vertex_data["label"][0]
#         Enhancer_Promoters[omic_type][label] = position


In [60]:
# from tqdm import tqdm
# # Create positional map
# positional_mapper = {
#     omic_type: {element: [] for element in omic_type_data}
#     for omic_type, omic_type_data in Enhancer_Promoters.items()
# }

# for col in tqdm(filtered_chunk.columns.to_list()[6:]):

#     chrom, interval_str = col.split(":")
#     start_str, end_str = interval_str.split("-")
#     start = int(start_str)
#     end = int(end_str)

#     for omic_type, omic_type_data in Enhancer_Promoters.items():
#         for element, position in omic_type_data.items():
#             same_chr = position["chr"] == chrom

#             # full containment
#             inside_region = (
#                 start >= position["start"] and end <= position["end"]
#             )

#             if same_chr and inside_region:
#                 positional_mapper[omic_type][element].append(col)
        

In [61]:
import sys
from pathlib import Path

PROJECT_ROOT = Path().resolve().parent
sys.path.append(str(PROJECT_ROOT))
from modules.utils.utils import load_json

positional_mapper = load_json("artifacts/positional_mapper.json")

In [62]:
import os
import pandas as pd
from typing import Dict, List, Optional


def flatten_required_columns_for_one_type(
    element_map: Dict[str, List[str]]
) -> List[str]:
    required_cols = set()
    for _, methylation_cols in element_map.items():
        required_cols.update(methylation_cols)
    return sorted(required_cols)


def build_output_column_map_for_one_type(
    omic_type: str,
    element_map: Dict[str, List[str]],
    prefix_with_omic_type: bool = False,
) -> Dict[str, List[str]]:
    """
    Build:
        output_column_name -> source methylation columns
    """
    output_map = {}

    safe_omic_type = omic_type.replace("/", "_").replace(" ", "_")

    for element_id, methylation_cols in element_map.items():
        if len(methylation_cols) == 0:
            continue

        if prefix_with_omic_type:
            output_col = f"{safe_omic_type}__{element_id}"
        else:
            output_col = str(element_id)

        output_map[output_col] = methylation_cols

    return output_map


def aggregate_methylation_for_one_regulatory_type(
    input_path: str,
    omic_type: str,
    element_map: Dict[str, List[str]],
    output_path: str,
    sample_id_col: Optional[str] = None,
    label_col: Optional[str] = None,
    extra_cols_to_keep: Optional[List[str]] = None,
    chunksize: int = 10,
    prefix_with_omic_type: bool = False,
) -> None:
    """
    Aggregate methylation values for one regulatory type only
    and save to one file.
    """
    if extra_cols_to_keep is None:
        extra_cols_to_keep = []

    output_col_to_source_cols = build_output_column_map_for_one_type(
        omic_type=omic_type,
        element_map=element_map,
        prefix_with_omic_type=prefix_with_omic_type,
    )

    required_methylation_cols = flatten_required_columns_for_one_type(element_map)

    metadata_cols = []
    if sample_id_col is not None:
        metadata_cols.append(sample_id_col)
    if label_col is not None:
        metadata_cols.append(label_col)
    metadata_cols.extend(extra_cols_to_keep)

    full_header = pd.read_csv(input_path, nrows=0).columns.tolist()
    available_cols = set(full_header)

    metadata_cols = [col for col in metadata_cols if col in available_cols]
    required_methylation_cols = [
        col for col in required_methylation_cols if col in available_cols
    ]

    usecols = metadata_cols + required_methylation_cols

    if len(required_methylation_cols) == 0:
        print(f"[SKIP] No mapped methylation columns found for {omic_type}")
        return

    first_chunk = True

    for chunk_idx, chunk in tqdm(enumerate(
        pd.read_csv(input_path, usecols=usecols, chunksize=chunksize, skip_blank_lines=False, keep_default_na=False),
        start=1,
    )):
        print(f"[{omic_type}] Processing chunk {chunk_idx} with shape {chunk.shape}")

        if len(metadata_cols) > 0:
            aggregated_chunk = chunk[metadata_cols].copy()
        else:
            aggregated_chunk = pd.DataFrame(index=chunk.index)

        for output_col, source_cols in output_col_to_source_cols.items():
            existing_source_cols = [col for col in source_cols if col in chunk.columns]

            if len(existing_source_cols) == 0:
                aggregated_chunk[output_col] = pd.NA
            elif len(existing_source_cols) == 1:
                aggregated_chunk[output_col] = chunk[existing_source_cols[0]]
            else:
                aggregated_chunk[output_col] = chunk[existing_source_cols].mean(axis=1)

        if first_chunk:
            aggregated_chunk.to_csv(output_path, index=False)
            first_chunk = False
        else:
            aggregated_chunk.to_csv(output_path, mode="a", header=False, index=False)

    print(f"[DONE] Saved {omic_type} aggregated matrix to: {output_path}")


def aggregate_methylation_to_separate_regulatory_files(
    input_path: str,
    positional_mapper: Dict[str, Dict[str, List[str]]],
    output_dir: str,
    sample_id_col: Optional[str] = None,
    label_col: Optional[str] = None,
    extra_cols_to_keep: Optional[List[str]] = None,
    chunksize: int = 100,
    prefix_with_omic_type: bool = False,
) -> None:
    """
    Save separate aggregated files for each regulatory type:
    - Enhancer
    - Promoter
    - Promoter/Enhancer
    """
    os.makedirs(output_dir, exist_ok=True)

    for omic_type, element_map in positional_mapper.items():
        safe_omic_type = omic_type.replace("/", "_").replace(" ", "_")
        output_path = os.path.join(
            output_dir,
            f"methylation_{safe_omic_type}_aggregated.csv",
        )

        aggregate_methylation_for_one_regulatory_type(
            input_path=input_path,
            omic_type=omic_type,
            element_map=element_map,
            output_path=output_path,
            sample_id_col=sample_id_col,
            label_col=label_col,
            extra_cols_to_keep=extra_cols_to_keep,
            chunksize=chunksize,
            prefix_with_omic_type=prefix_with_omic_type,
        )

In [63]:
input_path = "../data/ROSMAP/methylation_data_locations.csv"
output_dir = "../data/ROSMAP/methylation_regulatory_split"

aggregate_methylation_to_separate_regulatory_files(
    input_path=input_path,
    positional_mapper=positional_mapper,
    output_dir=output_dir,
    sample_id_col="Sample ID",   # change if needed
    label_col="Diagnosis",          # change if needed
    extra_cols_to_keep=["Gender", "Race", "PMI", "Braak"],
    chunksize=10,
    prefix_with_omic_type=False,
)

0it [00:00, ?it/s]

[Enhancer] Processing chunk 1 with shape (10, 889)


1it [00:04,  4.63s/it]

[Enhancer] Processing chunk 2 with shape (10, 889)


2it [00:09,  4.61s/it]

...

[Promoter/Enhancer] Processing chunk 15 with shape (10, 9718)


15it [14:43, 57.67s/it]

[Promoter/Enhancer] Processing chunk 16 with shape (5, 9718)


16it [15:42, 58.91s/it]

[DONE] Saved Promoter/Enhancer aggregated matrix to: ../data/ROSMAP/methylation_regulatory_split/methylation_Promoter_Enhancer_aggregated.csv


In [23]:
# get protiens enzymes list
from tqdm import tqdm
reactions = []

for vertex, vertex_data in graph_json["vertices"].items():
    if vertex_data["omic_type"] == "R":
        reactions.append(vertex)
enzymes = []

for edge in tqdm(graph_json["edges"]):
    if (edge["start_vertex"] in reactions) or (edge["end_vertex"] in reactions):
        enzymes.append(edge["start_vertex"])


100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 5121818/5121818 [03:45<00:00, 22760.39it/s]


In [25]:
# 
import sys
from pathlib import Path

PROJECT_ROOT = Path().resolve().parent
sys.path.append(str(PROJECT_ROOT))
from modules.utils.utils import load_json, save_json

save_json("../artifacts/enzymes.json", {"enzymes": enzymes})

In [29]:
save_json("../artifacts/enzymes.json", enzymes)

In [30]:
import os
import json

output_path = "../artifacts/enzymes.json"

# Ensure directory exists
os.makedirs(os.path.dirname(output_path), exist_ok=True)

# Save JSON
with open(output_path, "w") as f:
    json.dump(enzymes, f, indent=4)